In [34]:
import os
import sys
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import matplotlib

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.lines as mlines
from matplotlib.colors import Normalize
from matplotlib.gridspec import GridSpec
from matplotlib.path import Path as MplPath
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import re
from tools_eg import *

In [7]:
BASE = Path("/Users/emmagurcan/Documents/France/ENS/Speich/WHIRLS")
DIR_SADCP = BASE / "data/SADCP"
SADCP_FILES = {"OS150": DIR_SADCP / "WHIRLS_NB150_TOT_osite.nc",   # full cruise (README)
               "OS38": DIR_SADCP / "WHIRLS_BB38_TOT_osite.nc"}
DIR_VARDYN = BASE / "data/VARDYN"
DIR_LADCP = BASE / "data/LADCP/traitement_avec_SADCP"
DIR_SSH = BASE / "data/SSH"
DIR_EDDIES = {"TOEDDIES_better_ADT": BASE / "data/TOEDDIES/TOEDDIES_better_ADT",
              "TOEDDIES": BASE / "data/TOEDDIES/TOEDDIES"}
FNAME_BATHY = BASE / "data/bathymetry/etopo2.nc"
DIR_TRACKS = BASE / "eddies"
DIR_CODE = BASE / "analysis"          # where ladcp_tools.py lives
DIR_EDDIES_BETTER = DIR_EDDIES["TOEDDIES_better_ADT"]
DIR_EDDIES_STD = DIR_EDDIES["TOEDDIES"]
OUT_DIR = BASE / "figures/"
SADCP_TRACK_CSV = BASE / "figures/sadcp_vs_geostrophy/sadcp_vs_geostrophy_track.csv"

In [8]:
CRUISE_START = "2026-06-28"
CRUISE_END = "2026-07-27T23:59:59"

AVG_WINDOW = "3h"          # SADCP time averaging (reduces ping noise, not tides/inertial)
MIN_COVERAGE = 0.25        # keep a window if >= 25% of its expected ensembles are valid
                           # (expected count comes from each file's own sampling interval)
MIN_LAYER_FRAC = 0.5       # 0-100 m mean needs >= 50% of the bins that normally hold data
MAX_ADT_GAP = pd.Timedelta(hours=36)  # no geostrophy if the nearest ADT map is further away
Z_SURF = 30.0              # m — "near-surface" level
Z_LAYER = (50.0, 100.0)     # m — layer average
Z_SPLIT = 150.0            # m — use OS150 above, OS38 below for depth profiles
DEPTHS_SECTION = [30, 50, 75, 100, 150, 200, 300, 500] 
MIN_SPEED_ANGLE = 0.10     # m/s — angle differences below this speed are unreliable
ARROW_EVERY = "3h"         # arrow thinning on static maps
ARROW_SCALE = 2.0          # m/s per inch of arrow
ANIM_STEP = "2h"           # one animation frame every ANIM_STEP
ANIM_LEVEL = "layer"       # "layer" (0-100 m mean) or "surf" (30 m) for the ship arrows
ANIM_FPS = 6
ADD_LAND = os.environ.get("WHIRLS_NO_LAND") is None

G = 9.81
OMEGA = 7.2921e-5
R_EARTH = 6371e3


In [9]:
sadcp_all = {}
for inst in ["OS150", "OS38"]:
    s = cascade_add_time(xr.open_dataset(SADCP_FILES[inst]))
    s = s.sortby("time")
    s = s.isel(time=~s.indexes["time"].duplicated())
    sadcp_all[inst] = s

In [11]:
sh = sadcp_window("OS150", sadcp=cascade_to_window_format(sadcp_all["OS150"], pgood_min=50))
dp = sadcp_window("OS38",  sadcp=cascade_to_window_format(sadcp_all["OS38"],  pgood_min=50))

Calculating Mean
  ! PGOOD_ADCP is all 0 in this file -> PGOOD mask skipped
Calculating Mean


In [14]:
sh.keys()

dict_keys(['time', 'lon', 'lat', 'z', 'u', 'v'])

In [ ]:
# ---- ADT over the cruise area
x0 = float(np.nanmin(sh['lon'])) - 1.5
x1 = float(np.nanmax(sh['lon'])) + 1.5
y0 = float(np.nanmin(sh['lat'])) - 1.5
y1 = float(np.nanmax(sh['lat'])) + 1.5
EXTENT = [x0 + 0.5, x1 - 0.5, y0 + 0.5, y1 - 0.5]
adt_ds = load_adt(bbox=(x0, x1, y0, y1))

adt_days = pd.DatetimeIndex(pd.to_datetime(adt_ds.time.values)).floor("D")
missing = pd.date_range(CRUISE_START, pd.Timestamp(CRUISE_END).floor("D")).difference(adt_days)
print(f"ADT maps: {adt_days.min():%Y-%m-%d} -> {adt_days.max():%Y-%m-%d} ({len(adt_days)} days)")

In [ ]:
def date_from_name_cmems(f):
    m = re.search(r"(\d{4})-?(\d{2})-?(\d{2})", Path(f).name)
    return pd.Timestamp(f"{m.group(1)}-{m.group(2)}-{m.group(3)}")

def load_adt_cmems(dir_ssh, t0, t1, bbox=None, pad_days=1):
    files = sorted(os.listdir(DIR_SSH))

    # Select cruise dates
    days = pd.date_range(pd.Timestamp(t0) - pd.Timedelta(days=1),
                            pd.Timestamp(t1) + pd.Timedelta(days=1), freq="D")
    tags = {d.strftime("%Y-%m-%d") for d in days} | {d.strftime("%Y%m%d") for d in days}
    picked = [f for f in files if any(t in Path(f).name for t in tags)] or files
    dsets = []
    for f in picked:
        d = xr.open_dataset(dir_ssh / f)

        # fix the time for the daily files 
        if d.sizes.get("time", 1) == 1:
            if "time" in d.dims:
                d = d.isel(time=0, drop=True)
            elif "time" in d.coords:
                d = d.drop_vars("time")
            d = d.expand_dims(time=[date_from_name_cmems(f)])

        # load and close 
        dsets.append(d.load())
        d.close()

    # combine into 1 dataset along time axis
    ds = xr.concat(dsets, "time")
    ds = ds.assign_coords(time=np.asarray(ds.time.values, dtype="datetime64[ns]")).sortby("time")
    ds = ds.isel(time=~pd.Index(ds.time.values).duplicated())
    ds = ds.assign_coords(longitude=xr.where(ds.longitude > 180, ds.longitude - 360, ds.longitude)).sortby(["latitude", "longitude"])
    if bbox is not None:
        x0, x1, y0, y1 = bbox
        ds = ds.sel(longitude=slice(x0, x1), latitude=slice(y0, y1))
    return ds.sel(time=slice(days[0], days[-1] + pd.Timedelta(hours=23)))




def load_vardyn(dir_vardyn, t0, t1, bbox=None, pad_days=0):
    root = Path(dir_vardyn)
    folders = sorted(p for p in root.iterdir()
                     if p.is_dir() and not p.name.startswith("."))

    days = pd.date_range(pd.Timestamp(t0).normalize() - pd.Timedelta(days=pad_days),
                         pd.Timestamp(t1).normalize() + pd.Timedelta(days=pad_days), freq="D")
    tags = {d.strftime("y%Ym%md%d") for d in days}

    all_picked = []
    for folder in folders:
        files = sorted(p for p in folder.glob("*.nc") if not p.name.startswith("."))
        all_picked.extend(f for f in files if any(t in f.name for t in tags))
        
    return all_picked

# ds_cmems = load_adt_cmems(DIR_SSH, CRUISE_START, CRUISE_END,bbox=extent)
ds_vardyn = load_vardyn(DIR_VARDYN, CRUISE_START, CRUISE_END, bbox=extent)

VARDYN: 157 file(s), 2026-06-28 → 2026-07-27


In [39]:
ds['time']

<xarray.DataArray 'time' (time: 31)> Size: 248B
array(['2026-06-27T00:00:00.000000000', '2026-06-28T00:00:00.000000000',
       '2026-06-29T00:00:00.000000000', '2026-06-30T00:00:00.000000000',
       '2026-07-01T00:00:00.000000000', '2026-07-02T00:00:00.000000000',
       '2026-07-03T00:00:00.000000000', '2026-07-04T00:00:00.000000000',
       '2026-07-05T00:00:00.000000000', '2026-07-06T00:00:00.000000000',
       '2026-07-07T00:00:00.000000000', '2026-07-08T00:00:00.000000000',
       '2026-07-09T00:00:00.000000000', '2026-07-10T00:00:00.000000000',
       '2026-07-11T00:00:00.000000000', '2026-07-12T00:00:00.000000000',
       '2026-07-13T00:00:00.000000000', '2026-07-14T00:00:00.000000000',
       '2026-07-15T00:00:00.000000000', '2026-07-16T00:00:00.000000000',
       '2026-07-17T00:00:00.000000000', '2026-07-18T00:00:00.000000000',
       '2026-07-19T00:00:00.000000000', '2026-07-20T00:00:00.000000000',
       '2026-07-21T00:00:00.000000000', '2026-07-22T00:00:00.000000000',
       '2026-07-23T00:00:00.000000000', '2026-07-24T00:00:00.000000000',
       '2026-07-25T00:00:00.000000000', '2026-07-26T00:00:00.000000000',
       '2026-07-27T00:00:00.000000000'], dtype='datetime64[ns]')
Coordinates:
  * time     (time) datetime64[ns] 248B 2026-06-27 2026-06-28 ... 2026-07-27

In [ ]:
files = sorted(os.listdir(DIR_SSH))

# Select cruise dates
days = pd.date_range(pd.Timestamp(t0) - pd.Timedelta(days=1),
                         pd.Timestamp(t1) + pd.Timedelta(days=1), freq="D")
tags = {d.strftime("%Y-%m-%d") for d in days} | {d.strftime("%Y%m%d") for d in days}
picked = [f for f in files if any(t in Path(f).name for t in tags)] or files

In [28]:
picked

['2026-06-27_cmems_data.nc',
 '2026-06-28_cmems_data.nc',
 '2026-06-29_cmems_data.nc',
 '2026-06-30_cmems_data.nc',
 '2026-07-01_cmems_data.nc',
 '2026-07-02_cmems_data.nc',
 '2026-07-03_cmems_data.nc',
 '2026-07-04_cmems_data.nc',
 '2026-07-05_cmems_data.nc',
 '2026-07-06_cmems_data.nc',
 '2026-07-07_cmems_data.nc',
 '2026-07-08_cmems_data.nc',
 '2026-07-09_cmems_data.nc',
 '2026-07-10_cmems_data.nc',
 '2026-07-11_cmems_data.nc',
 '2026-07-12_cmems_data.nc',
 '2026-07-13_cmems_data.nc',
 '2026-07-14_cmems_data.nc',
 '2026-07-15_cmems_data.nc',
 '2026-07-16_cmems_data.nc',
 '2026-07-17_cmems_data.nc',
 '2026-07-18_cmems_data.nc',
 '2026-07-19_cmems_data.nc',
 '2026-07-20_cmems_data.nc',
 '2026-07-21_cmems_data.nc',
 '2026-07-22_cmems_data.nc',
 '2026-07-23_cmems_data.nc',
 '2026-07-24_cmems_data.nc',
 '2026-07-25_cmems_data.nc',
 '2026-07-26_cmems_data.nc',
 '2026-07-27_cmems_data.nc']

In [20]:
files

['.DS_Store',
 '2026-05-11_cmems_data.nc',
 '2026-05-12_cmems_data.nc',
 '2026-05-13_cmems_data.nc',
 '2026-05-14_cmems_data.nc',
 '2026-05-15_cmems_data.nc',
 '2026-05-16_cmems_data.nc',
 '2026-05-17_cmems_data.nc',
 '2026-05-18_cmems_data.nc',
 '2026-05-19_cmems_data.nc',
 '2026-05-20_cmems_data.nc',
 '2026-05-21_cmems_data.nc',
 '2026-05-22_cmems_data.nc',
 '2026-05-23_cmems_data.nc',
 '2026-05-24_cmems_data.nc',
 '2026-05-25_cmems_data.nc',
 '2026-05-26_cmems_data.nc',
 '2026-05-27_cmems_data.nc',
 '2026-05-28_cmems_data.nc',
 '2026-05-29_cmems_data.nc',
 '2026-05-30_cmems_data.nc',
 '2026-05-31_cmems_data.nc',
 '2026-06-01_cmems_data.nc',
 '2026-06-02_cmems_data.nc',
 '2026-06-03_cmems_data.nc',
 '2026-06-04_cmems_data.nc',
 '2026-06-05_cmems_data.nc',
 '2026-06-06_cmems_data.nc',
 '2026-06-07_cmems_data.nc',
 '2026-06-08_cmems_data.nc',
 '2026-06-09_cmems_data.nc',
 '2026-06-10_cmems_data.nc',
 '2026-06-11_cmems_data.nc',
 '2026-06-12_cmems_data.nc',
 '2026-06-13_cmems_data.nc',


In [25]:
ds = xr.open_dataset(DIR_SSH / files[1])

In [26]:
ds

<xarray.Dataset> Size: 675kB
Dimensions:    (time: 1, latitude: 156, longitude: 180)
Coordinates:
  * latitude   (latitude) float32 624B -44.44 -44.31 -44.19 ... -25.19 -25.06
  * longitude  (longitude) float32 720B 5.812 5.938 6.062 ... 27.94 28.06 28.19
  * time       (time) datetime64[ns] 8B 2026-05-10
Data variables:
    adt        (time, latitude, longitude) float64 225kB ...
    ugos       (time, latitude, longitude) float64 225kB ...
    vgos       (time, latitude, longitude) float64 225kB ...
Attributes: (12/45)
    Conventions:                     CF-1.6
    Metadata_Conventions:            Unidata Dataset Discovery v1.0
    cdm_data_type:                   Grid
    comment:                         Sea Surface Height measured by Altimetry...
    contact:                         servicedesk.cmems@mercator-ocean.eu
    creator_email:                   servicedesk.cmems@mercator-ocean.eu
    ...                              ...
    time_coverage_duration:          P1D
    time_coverage_end:               2024-11-30T12:00:00Z
    time_coverage_resolution:        P1D
    time_coverage_start:             2024-11-29T12:00:00Z
    title:                           NRT merged all satellites Global Ocean G...
    copernicusmarine_version:        2.4.0